In [ ]:
import pandas as pd

## 모델 실행 3: 상위 MMSI

MMSI가 35개 이상인 데이터 실행: 파일명 0630_tbl_ais_his_mmsi_35.csv

In [ ]:
# aisdfmmsi = pd.read_csv('0630_tbl_ais_his_dest_1005_mmsi_25.csv')
# aisdfmmsi = pd.read_csv('0630_tbl_ais_his_dest_1005_mmsi_30.csv')
# aisdfmmsi = pd.read_csv('0630_tbl_ais_his_dest_1005_mmsi_15.csv')
aisdfmmsi = pd.read_csv('0630_tbl_ais_his_dest_1005_mmsi_35.csv')

aisdfmmsi.head()

In [ ]:
aisdfmmsi['mmsi'].value_counts(ascending=True)

In [ ]:
aisdfmmsi.isnull().sum()

In [ ]:
aisdfmmsi1 = aisdfmmsi.drop(['ata', 'dest1', 'lat', 'lon','latlng1','rot'], axis=1)
aisdfmmsi1

In [ ]:
aisdfmmsi1 = aisdfmmsi1.fillna(aisdfmmsi1.mean())
aisdfmmsi1.isnull().sum()

In [ ]:
print(len(aisdfmmsi1['mmsi'].unique()))
print(len(aisdfmmsi1['ataport'].unique()))
k = sorted(aisdfmmsi1['ataport'].unique())
k

In [ ]:
aismmsidest = aisdfmmsi1['ataport']

aisdfmmsi1 = aisdfmmsi1.drop('ataport', axis=1)
aisdfmmsi1.head()

In [ ]:
# aisdfmmsi1 = aisdfmmsi1.drop({'mmsi','eta(unixTime)','utctime','navi','hdg','check'}, axis =1)
aisdfmmsi1 = aisdfmmsi1.drop({'eta(unixTime)','utctime','navi','hdg','check'}, axis =1)

# aisdfmmsi1 = aisdfmmsi1.drop({'eta(unixTime)','utctime','navi','hdg'}, axis =1) #mmsi 특징에 포함
aisdfmmsi1.head()

In [ ]:
# 정규화: Scale
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler()
# aisdfmmsi2 = aisdfmmsi1.copy()
# aisdfmmsi2[:] = scaler.fit_transform(aisdfmmsi2[:])
# aisdfmmsi2.head()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import model_selection

# 데이터를 로딩하고 학습데이타와 테스트 데이터 분리
X1 = aisdfmmsi1
# X1 = aisdfmmsi2
y1 = aismmsidest

X_train, X_test, y_train, y_test = model_selection.train_test_split(X1, y1, test_size=0.20, random_state=42)

# X_train, X_test, y_train, y_test = train_test_split(X1, y1, 
#                                                     test_size=0.2, random_state=121)

dtree = DecisionTreeClassifier()

### parameter 들을 dictionary 형태로 설정
# parameters = {'max_depth':[1,10,30, 50, 100, 150], 'min_samples_split':[2,4, 5, 7, 10, 20]}
parameters = {'max_depth':[100, 150, 200, 300, 400, 500], 'min_samples_split':[10, 20, 50, 100]}

# param_grid의 하이퍼 파라미터들을 3개의 train, test set fold 로 나누어서 테스트 수행 설정.  
### refit=True 가 default 임. True이면 가장 좋은 파라미터 설정으로 재 학습 시킴.  
grid_dtree = GridSearchCV(dtree, param_grid=parameters, cv=3, refit=True)

# Train 데이터로 param_grid의 하이퍼 파라미터들을 순차적으로 학습/평가 .
grid_dtree.fit(X_train, y_train)

# GridSearchCV 결과 추출하여 DataFrame으로 변환
scores_df = pd.DataFrame(grid_dtree.cv_results_)
scores_df[['params', 'mean_test_score', 'rank_test_score', \
           'split0_test_score', 'split1_test_score', 'split2_test_score']]

In [ ]:
from sklearn import model_selection

X = aisdfmmsi1
# X = aisdfmmsi2 # 정규화
y = aismmsidest

train_x, test_x, train_y, test_y = model_selection.train_test_split(X, y, test_size=0.20, random_state=42)
print(train_x.shape, test_x.shape, train_y.shape, test_y.shape) # 데이터 개수 확인
print(train_x.head())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score # 정확도 함수

rclf = RandomForestClassifier(n_estimators=10, max_depth=500,random_state=0)
rclf.fit(train_x,train_y)

predict1 = rclf.predict(test_x)
print(accuracy_score(test_y,predict1))

In [ ]:
import pickle
import joblib

saved_model_90 = pickle.dumps(rclf)

In [ ]:
clf_90 = pickle.loads(saved_model_90)
predict90_1 = clf_90.predict(test_x)
print(accuracy_score(test_y, predict90_1))

In [ ]:
joblib.dump(rclf,'rfmodel_1005_35.pkl')

In [ ]:
clf_90_from_joblib = joblib.load('rfmodel_1005_35.pkl')
predict_90_2 = clf_90_from_joblib.predict(test_x)
print(accuracy_score(test_y, predict_90_2))

In [ ]:
testdata = test_x.iloc[0:101]
onedata = testdata.iloc[0:5]
testpred = rclf.predict(testdata)
testreal = test_y.iloc[0:101]
print(testpred)
print(testreal)
print(onedata)

In [ ]:
# testdata = [[0.553991,0.583333,0.235294,0.072727,0.0,0.241111,0.031746,0.631579,0.000475]]
testpred = rclf.predict(onedata)
testreal = test_y.iloc[0:5]
print(testpred,",",testreal)
print(isinstance(testdata,list))
traindata = test_x.iloc[0:5]
print(traindata)

In [ ]:
print(len(test_x))

In [ ]:
test_x.to_csv("test_y.csv")

In [ ]:
predict90_1.to_csv("res_y.csv")

In [ ]:
testdata = test_x.iloc[0:5]
print(testdata)
for col in train_x.columns:
    print(col)

testdata = [[124,21,60,413523210,4.0,0.0,8.5,12,33,2.48]]
# testdata = [[0.553991,0.583333,0.235294,0.072727,0.0,0.241111,0.031746,0.631579,0.000475]]
# testdata = [[0.58216,0.583333,0.235294,0.57166,0.90909,0.0,0.023611,0.031746,0.631579,0.000129]]
testpred = rclf.predict(testdata)
testreal = test_y.iloc[1:2]
print(testpred,",",testreal)
# print(isinstance(testdata,list))
# traindata = test_x.iloc[1:4]
# print(traindata)